# Gemini Structured Output

LLM의 답변을 다른 코드에서 사용할 수 있도록 정해진 JSON 구조로 받는 방법을 배웁니다.

## 실습 환경 준비

`.env` 파일에 `GEMINI_API_KEY`가 설정되어 있어야 합니다.  

`pip install pydantic`

In [1]:
import json
import os
from pprint import pprint
from typing import Literal

from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, ConfigDict, Field, ValidationError

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
print("준비 완료 / 사용 모델:", model)

준비 완료 / 사용 모델: gemini-3.6-flash


## 일반 텍스트 응답의 한계

사람이 읽기에는 자연스러운 문장이 좋지만, 프로그램은 답변에서 필요한 값을 다시 찾아야 합니다. 답변의 표현이나 순서가 달라지면 후속 코드가 처리하기 어려워집니다.

In [4]:
review = "기능은 편리하지만 앱 실행이 느려요."

plain = client.interactions.create(
    model=model,
    input=f"다음 후기를 감정과 요약으로 분석해줘.\n\n{review}",
    store=False,
)
print(plain.output_text)

제시해주신 후기에 대한 감정과 요약 분석 결과입니다.

---

### **1. 감정 분석**
* **감정:** **복합적 (혼합)** — *긍정과 부정이 공존함*
* **세부 분석:**
  * **긍정적 측면:** 기능의 유용성과 편리함에 대한 만족 **(편리함)**
  * **부정적 측면:** 앱 실행 속도가 느린 점에 대한 아쉬움 및 답답함 **(불편함)**
* **종합 평가:** 앱의 제공 기능에 대해서는 호감을 느끼고 있으나, 성능(속도) 면에서 불만을 느끼고 있는 **'중립/개선 요구'** 상태입니다.

---

### **2. 요약**
* **한 줄 요약:** **기능은 편리하지만 앱 실행 속도가 느림.**
* **핵심 키워드:** 편리한 기능, 느린 실행 속도, 성능 개선 필요


## Pydantic으로 원하는 응답 구조 정의하기

### Pydantic이란?

Pydantic은 Python의 타입 힌트를 이용해 데이터의 구조를 정의하고 검증하는 라이브러리입니다. 일반 class처럼 모델을 만들지만, 전달받은 값이 선언한 자료형과 조건에 맞는지도 검사합니다.

이번에는 Gemini가 다음 구조로만 답하도록 요청합니다.

```json
{
  "sentiment": "positive | neutral | negative",
  "summary": "한 문장 요약",
  "keywords": ["핵심어"]
}
```

사람에게는 위의 JSON 예시만으로 구조를 설명할 수 있지만, Gemini와 프로그램에는 더 명확한 규칙이 필요합니다. 이 규칙을 표준 형식으로 표현한 것이 **JSON Schema**입니다. JSON Schema에는 필드 이름, 자료형, 필수 여부와 허용값 등이 들어갑니다.

JSON Schema를 dictionary로 직접 작성할 수도 있지만, 필드가 많아지면 모델 코드와 schema를 따로 관리해야 합니다. 여기서는 Pydantic 모델을 먼저 정의하고, 그 모델로부터 JSON Schema를 자동 생성합니다. 그러면 같은 모델을 Gemini 응답 형식 지정과 실제 응답 검증에 함께 사용할 수 있습니다.

In [2]:
class ReviewAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sentiment: Literal["positive", "neutral", "negative"]
    summary: str = Field(max_length=30)
    keywords: list[str] = Field(min_length=2, max_length=2)


# Pydantic 모델을 Gemini에 전달할 JSON Schema로 변환합니다.
review_schema = ReviewAnalysis.model_json_schema()
pprint(review_schema)

{'additionalProperties': False,
 'properties': {'keywords': {'items': {'type': 'string'},
                             'maxItems': 2,
                             'minItems': 2,
                             'title': 'Keywords',
                             'type': 'array'},
                'sentiment': {'enum': ['positive', 'neutral', 'negative'],
                              'title': 'Sentiment',
                              'type': 'string'},
                'summary': {'maxLength': 30,
                            'title': 'Summary',
                            'type': 'string'}},
 'required': ['sentiment', 'summary', 'keywords'],
 'title': 'ReviewAnalysis',
 'type': 'object'}


### Pydantic 모델 코드 읽기

- `BaseModel`: 데이터 모델을 만들기 위해 상속하는 Pydantic의 기본 class입니다.
- `Literal`: `sentiment`를 세 문자열 중 하나로 제한합니다.
- `Field`: 문자열 길이나 list 원소 개수 같은 추가 조건을 설정합니다.
- `ConfigDict(extra="forbid")`: 모델에 선언하지 않은 필드가 들어오면 검증에 실패하게 합니다.
- `model_json_schema()`: 모델의 타입과 조건을 JSON Schema dictionary로 변환합니다.

### 생성된 JSON Schema 읽기

- `type`: 값의 자료형입니다.
- `properties`: object 안에 들어갈 필드를 정의합니다.
- `enum`: 사용할 수 있는 값을 제한합니다.
- `items`: array 안에 들어갈 값의 자료형입니다.
- `required`: 반드시 있어야 하는 필드입니다.
- `additionalProperties: False`: 정의하지 않은 필드를 추가하지 못하게 합니다.

### 모델과 맞지 않는 JSON 검증하기

JSON 문법이 올바르더라도 값이 Pydantic 모델의 조건과 맞지 않으면 `ValidationError`가 발생합니다. Gemini에 요청을 보내기 전에 잘못된 JSON을 직접 넣어 검증 실패 상황을 확인해 봅니다.

In [5]:
invalid_json = json.dumps({
    "sentiment": "happy",       # 허용된 값이 아님
    "summary": "이 요약은 삼십 자 제한을 넘기기 위해 일부러 아주 길게 작성한 문장입니다.",
    "keywords": ["속도"],       # 원소가 2개가 아님
    "rating": 5,                   # 정의되지 않은 필드
}, ensure_ascii=False)

try:
    ReviewAnalysis.model_validate_json(invalid_json)
except ValidationError as error:
    print("검증 실패:")
    pprint(error.errors())

검증 실패:
[{'input': 5,
  'loc': ('rating',),
  'msg': 'Extra inputs are not permitted',
  'type': 'extra_forbidden',
  'url': 'https://errors.pydantic.dev/2.13/v/extra_forbidden'},
 {'ctx': {'expected': "'positive', 'neutral' or 'negative'"},
  'input': 'happy',
  'loc': ('sentiment',),
  'msg': "Input should be 'positive', 'neutral' or 'negative'",
  'type': 'literal_error',
  'url': 'https://errors.pydantic.dev/2.13/v/literal_error'},
 {'ctx': {'max_length': 30},
  'input': '이 요약은 삼십 자 제한을 넘기기 위해 일부러 아주 길게 작성한 문장입니다.',
  'loc': ('summary',),
  'msg': 'String should have at most 30 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.13/v/string_too_long'},
 {'ctx': {'actual_length': 1, 'field_type': 'List', 'min_length': 2},
  'input': ['속도'],
  'loc': ('keywords',),
  'msg': 'List should have at least 2 items after validation, not 1',
  'type': 'too_short',
  'url': 'https://errors.pydantic.dev/2.13/v/too_short'}]


## Structured Output 요청 보내기

### MIME type이란?

MIME type은 주고받는 데이터가 어떤 형식인지 나타내는 표준 문자열입니다. 예를 들어 일반 텍스트는 `text/plain`, HTML은 `text/html`, JSON은 `application/json`으로 표현합니다.

여기서 `mime_type="application/json"`은 Gemini에게 응답을 일반 문장이 아닌 **JSON 문법에 맞는 텍스트**로 만들라고 알려줍니다. 다만 MIME type은 JSON 내부에 어떤 필드가 있어야 하는지까지 정의하지 않습니다. 세부 구조와 제약 조건은 `schema`가 담당합니다.

`response_format`의 각 항목은 다음 역할을 합니다.

- `type: "text"`: 응답을 텍스트 출력으로 받습니다.
- `mime_type: "application/json"`: 그 텍스트의 형식을 JSON으로 제한합니다.
- `schema: review_schema`: JSON에 들어갈 필드, 자료형과 허용값을 제한합니다.

MIME type을 JSON으로 지정해도 Python이 곧바로 dictionary를 받는 것은 아닙니다. `output_text`에는 JSON 형식의 문자열이 들어오므로 이후에 `json.loads()`나 Pydantic으로 파싱해야 합니다.

In [8]:
structured = client.interactions.create(
    model=model,
    input=(
        "다음 후기를 분석하세요. summary는 30자 이내, "
        f"keywords는 2개만 작성하세요.\n\n[후기]\n{review}"
    ),
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": review_schema,
    },
    store=False,
)

print(structured.output_text)

{
  "sentiment": "neutral",
  "summary": "기능은 편리하나 앱 실행이 느림",
  "keywords": [
    "기능",
    "실행속도"
  ]
}


`output_text`는 JSON처럼 보이지만 아직 `str`입니다. dictionary로 사용하려면 파싱해야 합니다.

## JSON 문자열 파싱하기

`json.loads()`는 JSON 문자열을 Python 값으로 바꿉니다. 함수 이름의 `s`는 string을 뜻합니다.

In [9]:
parsed = json.loads(structured.output_text)

pprint(parsed)
print(type(parsed))
print("감정:", parsed["sentiment"])
print("요약:", parsed["summary"] )
print("키워드:", parsed["keywords"])

{'keywords': ['기능', '실행속도'],
 'sentiment': 'neutral',
 'summary': '기능은 편리하나 앱 실행이 느림'}
<class 'dict'>
감정: neutral
요약: 기능은 편리하나 앱 실행이 느림
키워드: ['기능', '실행속도']


## Pydantic으로 파싱과 검증하기

- 파싱: JSON 문법을 Python 값으로 변환합니다.
- 검증: 필드, 자료형과 허용값이 우리의 규칙에 맞는지 확인합니다.

JSON 문법이 맞더라도 필요한 필드가 없거나 값이 허용 범위를 벗어날 수 있으므로 후속 코드에서도 검사합니다. Pydantic의 `model_validate_json()`은 이 두 단계를 한 번에 수행합니다.

In [10]:
try:
    result = ReviewAnalysis.model_validate_json(structured.output_text)
except ValidationError as error:
    print("검증 실패:")
    pprint(error.errors())
else:
    print("검증 통과")
    print("Pydantic 모델:", result)
    print("감정:", result.sentiment)
    print("dictionary:")
    pprint(result.model_dump())

검증 통과
Pydantic 모델: sentiment='neutral' summary='기능은 편리하나 앱 실행이 느림' keywords=['기능', '실행속도']
감정: neutral
dictionary:
{'keywords': ['기능', '실행속도'],
 'sentiment': 'neutral',
 'summary': '기능은 편리하나 앱 실행이 느림'}


Structured Output은 응답의 **형식**을 안정적으로 만들지만, 분류와 요약의 **내용이 사실인지**까지 보장하지는 않습니다.

## 미니 실습: 문의 자동 분류기

앞의 코드를 수정해 고객 문의를 다음 구조로 분류하세요.

- `category`: `payment`, `account`, `delivery`, `other` 중 하나
- `urgency`: `low`, `medium`, `high` 중 하나
- `summary`: 한 문장 요약

입력: `결제가 두 번 됐어요. 하나를 취소해 주세요.`

In [11]:
# TODO: 문의 schema와 Structured Output 요청을 작성해 보세요.
customer_message = "결제가 두 번 됐어요. 하나를 취소해 주세요."

class ReviewAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid")

    category : Literal["payment", "account", "delivery", "other"]
    urgency : Literal["low", "medium", "high"]
    summary: str = Field(max_length=30)


# Pydantic 모델을 Gemini에 전달할 JSON Schema로 변환합니다.
review_schema = ReviewAnalysis.model_json_schema()


{'additionalProperties': False,
 'properties': {'category': {'enum': ['payment',
                                      'account',
                                      'delivery',
                                      'other'],
                             'title': 'Category',
                             'type': 'string'},
                'summary': {'maxLength': 30,
                            'title': 'Summary',
                            'type': 'string'},
                'urgency': {'enum': ['low', 'medium', 'high'],
                            'title': 'Urgency',
                            'type': 'string'}},
 'required': ['category', 'urgency', 'summary'],
 'title': 'ReviewAnalysis',
 'type': 'object'}


In [12]:
structured = client.interactions.create(
    model=model,
    input=(
        "다음 고객 문의를 분석하세요. summary는 30자 이내, "
        f"\n\n[문의]\n{customer_message}"
    ),
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": review_schema,
    },
    store=False,
)

print(structured.output_text)

{
  "category": "payment",
  "urgency": "high",
  "summary": "이중 결제 발생 취소 요청"
}
